# Turkish Whisper-Base LoRA Fine-Tuning (Google Colab)

Trains a lightweight LoRA adapter on `openai/whisper-base` using Google FLEURS Turkish (`tr_tr`, `train` split) on Colab's GPU so **none of the 1.8 GB dataset uses your local internet**.

At the end, it downloads **only `whisper-base-tr-lora.zip` (~3–6 MB)**.
Unzip it into `wsp/out/lora_adapter/` in your local repo and run:
```powershell
.venv\Scripts\python.exe wsp\compare_3way.py
```

In [ ]:
!pip install -q transformers peft accelerate datasets[audio] soundfile librosa evaluate jiwer

In [ ]:
import os, shutil, torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from datasets import load_dataset, Audio
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from peft import LoraConfig, get_peft_model

MODEL_ID = "openai/whisper-base"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# 1. Load FLEURS Turkish train split (downloaded on Google's servers, 0 local bandwidth)
ds = load_dataset("google/fleurs", "tr_tr", split="train")
ds = ds.cast_column("audio", Audio(sampling_rate=16000))
print("Train clips:", len(ds))

In [ ]:
processor = WhisperProcessor.from_pretrained(MODEL_ID, language="Turkish", task="transcribe")

def prepare_batch(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=16000
    ).input_features[0]
    txt = batch["raw_transcription"] or batch["transcription"]
    batch["labels"] = processor.tokenizer(txt).input_ids
    return batch

ds_proc = ds.map(prepare_batch, remove_columns=ds.column_names, num_proc=2)

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [ ]:
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.config.use_cache = False

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

args = Seq2SeqTrainingArguments(
    output_dir="./whisper-base-tr-lora-ckpt",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-3,
    warmup_steps=50,
    num_train_epochs=4,
    fp16=torch.cuda.is_available(),
    logging_steps=25,
    save_strategy="no",
    remove_unused_columns=False,
    label_names=["labels"],
    report_to="none",
)

trainer = Seq2SeqTrainer(
    args=args,
    model=model,
    train_dataset=ds_proc,
    data_collator=collator,
    processing_class=processor.feature_extractor,
)
trainer.train()

In [ ]:
OUT_DIR = "lora_adapter"
model.save_pretrained(OUT_DIR)
shutil.make_archive("whisper-base-tr-lora", "zip", OUT_DIR)
size_mb = os.path.getsize("whisper-base-tr-lora.zip") / 1024**2
print(f"Saved whisper-base-tr-lora.zip ({size_mb:.2f} MB)")

from google.colab import files
files.download("whisper-base-tr-lora.zip")